In [ ]:
# ====================== 1. Imports and setup ======================
print("="*80)
print("5分钟高频数据三分类：改进版")
print("预测未来12个5分钟，增加中性类别，确保序列在同一交易日内")
print("="*80)
print("导入所需库...")

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, roc_auc_score, classification_report,
    roc_curve, precision_recall_curve, auc
)
from torch.utils.data import TensorDataset, DataLoader
import os
import time
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr
from tqdm import tqdm
import json
from sklearn.utils.class_weight import compute_class_weight

# Configure fonts for Chinese display text.
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
mpl.rcParams['font.family'] = 'sans-serif'

# Set the random seed.
torch.manual_seed(42)
np.random.seed(42)

print("✓ 库导入和设置完成")

# Global parameters
SEQ_LEN = 60  # 60 five-minute bars = 5 hours
PRED_BARS = 12  # Predict the next 12 five-minute bars = 60 minutes.
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
BATCH_SIZE = 32
EPOCHS = 50
LR = 0.001
PATIENCE = 10
NEUTRAL_THRESHOLD = 0.0003  # default threshold for single-run checks
NUM_CLASSES = 3  # Three classes: down (0), neutral (1), up (2).

# Threshold sweep config (advisor priority)
SWEEP_THRESHOLDS = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]
SWEEP_MODELS = ['TCN', 'DLinear', 'LSTM']  # run all three models
SWEEP_EPOCHS = 10
SWEEP_PATIENCE = 4
USE_CLASS_WEIGHT_FOR_SWEEP = False  # keep off first; test threshold effect first


print(f"全局参数: 序列长度={SEQ_LEN}个5分钟, 预测窗口={PRED_BARS}个5分钟")
print(f"中性阈值: {NEUTRAL_THRESHOLD}")
print(f"分类: 下跌(<{-NEUTRAL_THRESHOLD}), 中性([-{NEUTRAL_THRESHOLD}, {NEUTRAL_THRESHOLD}]), 上涨(>{NEUTRAL_THRESHOLD})")


print(f"Sweep thresholds: {SWEEP_THRESHOLDS}")
print(f"Sweep models: {SWEEP_MODELS}")
print(f"Use class weight in sweep: {USE_CLASS_WEIGHT_FOR_SWEEP}")


In [ ]:
# ====================== 2. Preprocessing and feature engineering ======================
print("\n" + "="*80)
print("步骤2: 5分钟数据预处理和特征工程")
print("="*80)

# Reuse the earlier functions and load the preprocessed data here.
def load_cleaned_data():
    """Load the previously cleaned five-minute data."""
    file_path = 'EEM_5min_cleaned.csv'
    
    if not os.path.exists(file_path):
        print(f"❌ 文件不存在: {file_path}")
        return None
    
    print(f"加载清理后的数据: {file_path}")
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    
    # Ensure a DatetimeIndex.
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    print(f"数据形状: {df.shape}")
    print(f"时间范围: {df.index.min()} 到 {df.index.max()}")
    
    return df

# Load data.
eem_df = load_cleaned_data()

if eem_df is not None:
    print("\n数据前5行:")
    print(eem_df.head())
    
    print(f"\n数据统计:")
    print(f"总样本数: {len(eem_df)}")
    print(f"特征数量: {len(eem_df.columns)-1} (不包括LogReturn)")
    
    # Check data quality.
    print(f"\n数据质量检查:")
    print(f"缺失值数量: {eem_df.isnull().sum().sum()}")
    print(f"收益率均值: {eem_df['LogReturn'].mean():.6f}")
    print(f"收益率标准差: {eem_df['LogReturn'].std():.6f}")
    print(f"正收益率比例: {(eem_df['LogReturn'] > 0).mean():.4f}")
    
    # Calculate the return distribution.
    returns = eem_df['LogReturn'].values
    print(f"收益率分布:")
    print(f"  最小值: {returns.min():.6f}")
    print(f"  最大值: {returns.max():.6f}")
    print(f"  10%分位数: {np.percentile(returns, 10):.6f}")
    print(f"  25%分位数: {np.percentile(returns, 25):.6f}")
    print(f"  中位数: {np.percentile(returns, 50):.6f}")
    print(f"  75%分位数: {np.percentile(returns, 75):.6f}")
    print(f"  90%分位数: {np.percentile(returns, 90):.6f}")
    
    # Check whether the neutral threshold is appropriate.
    near_zero_ratio = np.mean(np.abs(returns) < NEUTRAL_THRESHOLD)
    print(f"\n收益率绝对值小于中性阈值({NEUTRAL_THRESHOLD})的比例: {near_zero_ratio:.4f}")
else:
    print("❌ 数据加载失败")

In [ ]:
# ====================== 3. Dataset split ======================
print("\n" + "="*80)
print("步骤3: 5分钟数据集划分")
print("="*80)

if eem_df is not None:
    # Split parameters
    total_size = len(eem_df)
    train_size = int(total_size * TRAIN_RATIO)
    val_size = int(total_size * VAL_RATIO)
    test_size = total_size - train_size - val_size
    
    # Split the data chronologically.
    train_df = eem_df.iloc[:train_size]
    val_df = eem_df.iloc[train_size:train_size+val_size]
    test_df = eem_df.iloc[train_size+val_size:]
    
    print(f"数据集划分:")
    print(f"训练集: {len(train_df)} 样本 ({TRAIN_RATIO*100:.0f}%)")
    print(f"验证集: {len(val_df)} 样本 ({VAL_RATIO*100:.0f}%)")
    print(f"测试集: {len(test_df)} 样本 ({100-TRAIN_RATIO*100-VAL_RATIO*100:.0f}%)")
    
    # Print date ranges.
    print(f"\n训练集时间: {train_df.index.min()} 到 {train_df.index.max()}")
    print(f"验证集时间: {val_df.index.min()} 到 {val_df.index.max()}")
    print(f"测试集时间: {test_df.index.min()} 到 {test_df.index.max()}")
else:
    print("❌ 无可用数据")

In [ ]:
# ====================== 4. Construct three-class sequences ======================
print("\n" + "="*80)
print("步骤4: 创建三分类序列")
print("="*80)

def is_same_trading_day(sequence_times):
    """Check that all timestamps in the sequence belong to the same trading day."""
    if len(sequence_times) == 0:
        return False
    
    first_date = sequence_times[0].date()
    for t in sequence_times:
        if t.date() != first_date:
            return False
    return True

def create_sequences_5min_multiclass(data, seq_len=60, pred_bars=12, neutral_threshold=0.0005):
    """
    Construct sequences from five-minute bars.
    Predict the direction of the mean return over the next pred_bars bars.
    Classes: down (0), neutral (1), up (2).
    """
    # Feature columns, excluding LogReturn
    feature_columns = [col for col in data.columns if col != 'LogReturn']
    print(f"特征列: {feature_columns}")
    print(f"中性阈值: {neutral_threshold}")
    
    X, y = [], []
    
    # Extract features and labels.
    features = data[feature_columns].values
    returns = data['LogReturn'].values
    indices = data.index
    
    # Counts
    down_count, neutral_count, up_count = 0, 0, 0
    cross_day_skipped = 0
    
    for i in range(seq_len, len(data) - pred_bars):
        # Check whether the sequence stays within one trading day.
        sequence_times = indices[i-seq_len:i]
        if not is_same_trading_day(sequence_times):
            cross_day_skipped += 1
            continue
        
        # Input sequence
        input_seq = features[i-seq_len:i]
        
        # Target: mean return over the next pred_bars five-minute bars.
        future_returns = returns[i:i+pred_bars]
        avg_return = np.mean(future_returns)
        
        # Three-class label
        if avg_return < -neutral_threshold:
            label = 0  # Down
            down_count += 1
        elif avg_return > neutral_threshold:
            label = 2  # Up
            up_count += 1
        else:
            label = 1  # Neutral
            neutral_count += 1
        
        X.append(input_seq)
        y.append(label)
    
    total = down_count + neutral_count + up_count
    print(f"\n序列创建统计:")
    print(f"  总序列数: {total}")
    print(f"  因跨交易日跳过的序列: {cross_day_skipped}")
    print(f"\n标签分布:")
    print(f"  下跌(0): {down_count} ({down_count/total*100:.1f}%)")
    print(f"  中性(1): {neutral_count} ({neutral_count/total*100:.1f}%)")
    print(f"  上涨(2): {up_count} ({up_count/total*100:.1f}%)")
    
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

def standardize_data_5min(train_features, val_features, test_features):
    """Standardize five-minute data."""
    scaler = StandardScaler()
    
    # Fit the scaler on training data only.
    train_features_scaled = scaler.fit_transform(train_features)
    val_features_scaled = scaler.transform(val_features)
    test_features_scaled = scaler.transform(test_features)
    
    return train_features_scaled, val_features_scaled, test_features_scaled, scaler

if eem_df is not None:
    print(f"序列长度: {SEQ_LEN}个5分钟")
    print(f"预测窗口: {PRED_BARS}个5分钟 (60分钟)")
    print(f"中性阈值: {NEUTRAL_THRESHOLD}")
    print(f"预测目标: 未来{PRED_BARS}个5分钟平均收益率三分类")
    
    # Construct sequences.
    print("\n创建训练集序列...")
    X_train_raw, y_train = create_sequences_5min_multiclass(train_df, SEQ_LEN, PRED_BARS, NEUTRAL_THRESHOLD)
    print(f"训练集序列形状: {X_train_raw.shape}, 标签形状: {y_train.shape}")
    
    print("\n创建验证集序列...")
    X_val_raw, y_val = create_sequences_5min_multiclass(val_df, SEQ_LEN, PRED_BARS, NEUTRAL_THRESHOLD)
    print(f"验证集序列形状: {X_val_raw.shape}, 标签形状: {y_val.shape}")
    
    print("\n创建测试集序列...")
    X_test_raw, y_test = create_sequences_5min_multiclass(test_df, SEQ_LEN, PRED_BARS, NEUTRAL_THRESHOLD)
    print(f"测试集序列形状: {X_test_raw.shape}, 标签形状: {y_test.shape}")
    
    # Flatten 3D data to 2D for standardization.
    train_samples, train_timesteps, train_features = X_train_raw.shape
    val_samples, val_timesteps, val_features = X_val_raw.shape
    test_samples, test_timesteps, test_features = X_test_raw.shape
    
    X_train_flat = X_train_raw.reshape(-1, train_features)
    X_val_flat = X_val_raw.reshape(-1, train_features)
    X_test_flat = X_test_raw.reshape(-1, train_features)
    
    # Standardize.
    print("\n标准化数据...")
    X_train_flat_scaled, X_val_flat_scaled, X_test_flat_scaled, scaler = standardize_data_5min(
        X_train_flat, X_val_flat, X_test_flat
    )
    
    # Reshape back to 3D.
    X_train = X_train_flat_scaled.reshape(train_samples, train_timesteps, train_features)
    X_val = X_val_flat_scaled.reshape(val_samples, val_timesteps, train_features)
    X_test = X_test_flat_scaled.reshape(test_samples, test_timesteps, train_features)
    
    print(f"\n标准化后数据形状:")
    print(f"训练集: {X_train.shape} -> 标签: {y_train.shape}")
    print(f"验证集: {X_val.shape} -> 标签: {y_val.shape}")
    print(f"测试集: {X_test.shape} -> 标签: {y_test.shape}")
    
    # Validate standardization.
    train_mean = X_train.mean()
    train_std = X_train.std()
    val_mean = X_val.mean()
    val_std = X_val.std()
    test_mean = X_test.mean()
    test_std = X_test.std()
    
    print(f"\n标准化验证:")
    print(f"训练集 - 均值: {train_mean:.4f}, 标准差: {train_std:.4f}")
    print(f"验证集 - 均值: {val_mean:.4f}, 标准差: {val_std:.4f}")
    print(f"测试集 - 均值: {test_mean:.4f}, 标准差: {test_std:.4f}")
    
    if abs(train_std - 1) > 0.1 or abs(val_std - 1) > 0.1 or abs(test_std - 1) > 0.1:
        print("⚠️ 警告: 标准化后标准差偏离1较多，可能存在数据分布变化")
else:
    print("❌ 无可用数据")

In [ ]:
# ====================== 5. Three-class model definitions ======================
print("\n" + "="*80)
print("步骤5: 三分类模型定义 - TCN, DLinear, LSTM")
print("="*80)

# ====================== TCN model ======================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, 
                                     dilation=dilation_size, padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

class TCNClassifierMulticlass(nn.Module):
    def __init__(self, input_size, seq_len, num_classes=3, num_channels=[32, 32, 32], kernel_size=3, dropout=0.2):
        super(TCNClassifierMulticlass, self).__init__()
        self.tcn = TemporalConvNet(input_size, num_channels, kernel_size, dropout)
        self.linear = nn.Linear(num_channels[-1], num_classes)
    def forward(self, x):
        x = x.transpose(1, 2)
        y = self.tcn(x)
        output = self.linear(y[:, :, -1])
        return output

# ====================== DLinear model ======================
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)
    def forward(self, x):
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x

class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)
    def forward(self, x):
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean

class DLinearClassifierMulticlass(nn.Module):
    def __init__(self, seq_len, num_features, num_classes=3, pred_len=12, kernel_size=25, dropout=0.2):
        super(DLinearClassifierMulticlass, self).__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_features = num_features
        self.decomposition = series_decomp(kernel_size)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.classifier = nn.Sequential(
            nn.Linear(num_features * pred_len, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        seasonal_init, trend_init = self.decomposition(x)
        trend_output = self.linear_trend(trend_init)
        seasonal_output = self.linear_seasonal(seasonal_init)
        output = trend_output + seasonal_output
        output = output.transpose(1, 2)
        output = output.reshape(output.size(0), -1)
        output = self.classifier(output)
        return output

# ====================== LSTM model ======================
class LSTMClassifierMulticlass(nn.Module):
    def __init__(self, input_size, num_classes=3, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMClassifierMulticlass, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        lstm_out = self.dropout(lstm_out[:, -1, :])
        output = self.linear(lstm_out)
        return output

# Initialize models.
if eem_df is not None:
    num_features = X_train.shape[2]
    print(f"特征数量: {num_features}")
    print(f"类别数量: {NUM_CLASSES}")
    
    # TCN model
    tcn_model = TCNClassifierMulticlass(
        input_size=num_features,
        seq_len=SEQ_LEN,
        num_classes=NUM_CLASSES,
        num_channels=[32, 32, 32],
        kernel_size=3,
        dropout=0.2
    )
    
    # DLinear model
    dlinear_model = DLinearClassifierMulticlass(
        seq_len=SEQ_LEN,
        num_features=num_features,
        num_classes=NUM_CLASSES,
        pred_len=PRED_BARS,
        kernel_size=25,
        dropout=0.2
    )
    
    # LSTM model
    lstm_model = LSTMClassifierMulticlass(
        input_size=num_features,
        num_classes=NUM_CLASSES,
        hidden_size=64,
        num_layers=2,
        dropout=0.2
    )
    
    print("\n模型初始化完成:")
    print(f"TCN参数数量: {sum(p.numel() for p in tcn_model.parameters() if p.requires_grad):,}")
    print(f"DLinear参数数量: {sum(p.numel() for p in dlinear_model.parameters() if p.requires_grad):,}")
    print(f"LSTM参数数量: {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")
    
    # Test the models.
    test_input = torch.randn(2, SEQ_LEN, num_features)
    with torch.no_grad():
        tcn_output = tcn_model(test_input)
        dlinear_output = dlinear_model(test_input)
        lstm_output = lstm_model(test_input)
        
    print(f"\n测试输入形状: {test_input.shape}")
    print(f"TCN输出形状: {tcn_output.shape} (logits)")
    print(f"DLinear输出形状: {dlinear_output.shape} (logits)")
    print(f"LSTM输出形状: {lstm_output.shape} (logits)")
    print(f"每个样本有{NUM_CLASSES}个类别的logits值")
else:
    print("❌ 无可用数据，跳过模型初始化")

In [ ]:
# ====================== 6. Three-class training and evaluation ======================
print("\n" + "="*80)
print("步骤6: 三分类训练和评估函数")
print("="*80)

def calculate_multiclass_metrics(predictions, targets, num_classes=3):
    """Calculate multiclass evaluation metrics."""
    # predictions contains probabilities or logits; targets contains class labels.
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    # Convert logits to probabilities if necessary.
    if predictions.ndim > 1 and predictions.shape[1] > 1:
        pred_probs = predictions
        pred_classes = np.argmax(predictions, axis=1)
    else:
        # If predictions already contains class labels
        pred_classes = predictions
        # Construct one-hot probabilities.
        pred_probs = np.eye(num_classes)[pred_classes]
    
    # Basic metrics
    accuracy = accuracy_score(targets, pred_classes)
    
    # Averaged multiclass metrics
    precision_macro = precision_score(targets, pred_classes, average='macro', zero_division=0)
    recall_macro = recall_score(targets, pred_classes, average='macro', zero_division=0)
    f1_macro = f1_score(targets, pred_classes, average='macro', zero_division=0)
    
    # Weighted metrics
    precision_weighted = precision_score(targets, pred_classes, average='weighted', zero_division=0)
    recall_weighted = recall_score(targets, pred_classes, average='weighted', zero_division=0)
    f1_weighted = f1_score(targets, pred_classes, average='weighted', zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(targets, pred_classes)
    
    # Calculate per-class metrics.
    precision_per_class = precision_score(targets, pred_classes, average=None, zero_division=0)
    recall_per_class = recall_score(targets, pred_classes, average=None, zero_division=0)
    f1_per_class = f1_score(targets, pred_classes, average=None, zero_division=0)
    
    # Calculate AUC using one-vs-rest.
    auc_scores = []
    for i in range(num_classes):
        try:
            auc_score = roc_auc_score((targets == i).astype(int), pred_probs[:, i])
            auc_scores.append(auc_score)
        except:
            auc_scores.append(0.5)
    auc_macro = np.mean(auc_scores)
    
    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
        'precision_weighted': precision_weighted, 'recall_weighted': recall_weighted, 'f1_weighted': f1_weighted,
        'precision_per_class': precision_per_class, 'recall_per_class': recall_per_class, 'f1_per_class': f1_per_class,
        'auc_macro': auc_macro, 'auc_per_class': auc_scores,
        'confusion_matrix': cm,
        'class_distribution': np.bincount(targets, minlength=num_classes)
    }

def safe_train_model_multiclass(model, model_name, train_loader, val_loader, 
                               num_classes=3, num_epochs=50, lr=0.001, patience=10, use_class_weight=True):
    """Train one multiclass model with numerical checks."""
    print(f"\n训练 {model_name} 模型...")
    print("-" * 50)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Calculate class weights for imbalance.
    # Optional class weighting: keep it off by default for threshold-effect analysis
    if use_class_weight:
        all_train_targets = []
        for _, y_batch in train_loader:
            all_train_targets.extend(y_batch.cpu().numpy())

        class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=all_train_targets)
        class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f"  Using class weights: {class_weights.detach().cpu().numpy()}")
    else:
        criterion = nn.CrossEntropyLoss()
        print("  Class weights disabled for threshold sweep.")
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_f1 = 0
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training phase
        model.train()
        train_loss, train_logits, train_targets = 0, [], []
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            outputs = model(X_batch)
            
            # Check outputs for NaN or infinity.
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                print(f"⚠️ 警告: 批次 {batch_idx} 输出包含NaN/Inf，跳过")
                continue
                
            loss = criterion(outputs, y_batch)
            
            # Check the loss for NaN.
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"⚠️ 警告: 批次 {batch_idx} 损失为NaN/Inf，跳过")
                continue
                
            loss.backward()
            
            # Clip gradients.
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            train_loss += loss.item()
            
            with torch.no_grad():
                train_logits.extend(outputs.cpu().numpy())
                train_targets.extend(y_batch.cpu().numpy())
        
        # Skip this epoch if there are no valid training batches.
        if len(train_logits) == 0:
            print(f"⚠️ 警告: 第{epoch+1}轮无有效训练数据，跳过")
            continue
            
        # Validation phase
        model.eval()
        val_loss, val_logits, val_targets = 0, [], []
        with torch.no_grad():
            for batch_idx, (X_batch, y_batch) in enumerate(val_loader):
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                
                # Check outputs for NaN.
                if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                    continue
                    
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                
                val_logits.extend(outputs.cpu().numpy())
                val_targets.extend(y_batch.cpu().numpy())
        
        # Skip this epoch if there are no valid validation batches.
        if len(val_logits) == 0 or len(val_targets) == 0:
            print(f"⚠️ 警告: 第{epoch+1}轮无有效验证数据，跳过")
            continue
        
        # Calculate metrics.
        train_metrics = calculate_multiclass_metrics(np.array(train_logits), np.array(train_targets), num_classes)
        val_metrics = calculate_multiclass_metrics(np.array(val_logits), np.array(val_targets), num_classes)
        
        # Schedule the learning rate.
        scheduler.step(val_loss / max(len(val_loader), 1))
        current_lr = optimizer.param_groups[0]['lr']
        
        epoch_time = time.time() - start_time
        
        # Print progress.
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{num_epochs} | 时间: {epoch_time:.1f}s | 学习率: {current_lr:.6f}')
            print(f'  损失: 训练{train_loss/len(train_loader):.4f} / 验证{val_loss/len(val_loader):.4f}')
            print(f'  准确率: 训练{train_metrics["accuracy"]:.4f} / 验证{val_metrics["accuracy"]:.4f}')
            print(f'  F1分数(加权): 训练{train_metrics["f1_weighted"]:.4f} / 验证{val_metrics["f1_weighted"]:.4f}')
            print(f'  AUC(宏观平均): {val_metrics["auc_macro"]:.4f}')
            
            # Display per-class F1 scores.
            print(f'  各类别F1:')
            for i in range(num_classes):
                class_name = ['下跌', '中性', '上涨'][i]
                print(f'    {class_name}: {val_metrics["f1_per_class"][i]:.4f}')
        
        # Early stopping
        current_val_f1 = val_metrics['f1_weighted']
        if current_val_f1 > best_val_f1:
            best_val_f1 = current_val_f1
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()
            torch.save(best_model_state, f'best_5min_{model_name.lower()}_multiclass.pth')
            print(f'  ✓ 保存最佳模型 (加权F1={best_val_f1:.4f})')
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f'\n⚠️  {model_name} 早停于第 {epoch+1} 轮')
            break
    
    # Load the best model.
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    print(f'\n✓  {model_name} 训练完成')
    print(f'   最佳验证加权F1: {best_val_f1:.4f}')
    
    return model, best_val_f1

def safe_evaluate_model_multiclass(model, data_loader, num_classes=3):
    """Evaluate a multiclass model with numerical checks."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)
    
    all_logits, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            
            # Check outputs for NaN.
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                continue
                
            all_logits.extend(outputs.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
    
    all_logits = np.array(all_logits, dtype=np.float32)
    all_targets = np.array(all_targets, dtype=np.int64)
    
    return all_logits, all_targets

print("✓ 三分类训练和评估函数定义完成")


In [ ]:
# ====================== 7. Prepare data loaders ======================
print("\n" + "="*80)
print("步骤7: 准备5分钟数据加载器")
print("="*80)

if eem_df is not None:
    # Convert to PyTorch tensors.
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)  # Multiclass labels require long integer tensors.
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)
    
    # Create data loaders.
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    print(f"数据加载器创建完成:")
    print(f"训练集批次: {len(train_loader)} (批次大小: {BATCH_SIZE})")
    print(f"验证集批次: {len(val_loader)}")
    print(f"测试集批次: {len(test_loader)}")
    
    # Validate the first batch.
    for X_batch, y_batch in train_loader:
        print(f"\n第一个训练批次:")
        print(f"  输入形状: {X_batch.shape}")
        print(f"  标签形状: {y_batch.shape}")
        
        # Label distribution
        class_names = ['下跌', '中性', '上涨']
        for i in range(NUM_CLASSES):
            count = (y_batch == i).sum().item()
            print(f"  {class_names[i]}: {count}")
        
        # Data range
        print(f"  输入范围: [{X_batch.min():.3f}, {X_batch.max():.3f}]")
        print(f"  输入均值: {X_batch.mean():.3f}, 标准差: {X_batch.std():.3f}")
        break
else:
    print("❌ 无可用数据")

In [ ]:
# ====================== 8. Threshold Sweep Training ======================
print("\n" + "="*80)
print("Step 8: Run neutral-threshold sweep (advisor suggestion)")
print("="*80)

if eem_df is not None:
    print(f"Threshold list: {SWEEP_THRESHOLDS}")
    print(f"Model list: {SWEEP_MODELS}")
    print(f"Use class weight: {USE_CLASS_WEIGHT_FOR_SWEEP}")

    def build_model_by_name(model_name, num_features):
        if model_name == 'TCN':
            return TCNClassifierMulticlass(
                input_size=num_features,
                seq_len=SEQ_LEN,
                num_classes=NUM_CLASSES,
                num_channels=[32, 32, 32],
                kernel_size=3,
                dropout=0.2,
            )
        if model_name == 'DLinear':
            return DLinearClassifierMulticlass(
                seq_len=SEQ_LEN,
                num_features=num_features,
                num_classes=NUM_CLASSES,
                pred_len=PRED_BARS,
                kernel_size=25,
                dropout=0.2,
            )
        if model_name == 'LSTM':
            return LSTMClassifierMulticlass(
                input_size=num_features,
                num_classes=NUM_CLASSES,
                hidden_size=64,
                num_layers=2,
                dropout=0.2,
            )
        raise ValueError(f"Unsupported model: {model_name}")

    def init_weights_safe(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight, gain=0.5)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
        elif isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight, gain=0.5)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
        elif isinstance(m, nn.LSTM):
            for name, param in m.named_parameters():
                if 'weight' in name:
                    nn.init.xavier_uniform_(param, gain=0.5)
                elif 'bias' in name:
                    nn.init.constant_(param, 0.01)

    def label_stats(y):
        total = len(y)
        down = int(np.sum(y == 0))
        neutral = int(np.sum(y == 1))
        up = int(np.sum(y == 2))
        return {
            'total': total,
            'down_count': down,
            'neutral_count': neutral,
            'up_count': up,
            'down_ratio': down / total if total > 0 else 0.0,
            'neutral_ratio': neutral / total if total > 0 else 0.0,
            'up_ratio': up / total if total > 0 else 0.0,
        }

    sweep_results = []
    sweep_class_balance = []

    for threshold in SWEEP_THRESHOLDS:
        print("\n" + "-" * 80)
        print(f"Running threshold = {threshold}")
        print("-" * 80)

        X_train_raw_thr, y_train_thr = create_sequences_5min_multiclass(train_df, SEQ_LEN, PRED_BARS, threshold)
        X_val_raw_thr, y_val_thr = create_sequences_5min_multiclass(val_df, SEQ_LEN, PRED_BARS, threshold)
        X_test_raw_thr, y_test_thr = create_sequences_5min_multiclass(test_df, SEQ_LEN, PRED_BARS, threshold)

        if len(y_train_thr) == 0 or len(y_val_thr) == 0 or len(y_test_thr) == 0:
            print(f"Skip threshold {threshold}: one split has zero samples.")
            continue

        train_stats = label_stats(y_train_thr)
        val_stats = label_stats(y_val_thr)
        test_stats = label_stats(y_test_thr)

        sweep_class_balance.extend([
            {'threshold': threshold, 'split': 'train', **train_stats},
            {'threshold': threshold, 'split': 'val', **val_stats},
            {'threshold': threshold, 'split': 'test', **test_stats},
        ])

        print(
            "Train class ratio -> "
            f"down={train_stats['down_ratio']:.3f}, "
            f"neutral={train_stats['neutral_ratio']:.3f}, "
            f"up={train_stats['up_ratio']:.3f}"
        )

        train_samples, train_timesteps, train_features = X_train_raw_thr.shape
        val_samples, val_timesteps, val_features = X_val_raw_thr.shape
        test_samples, test_timesteps, test_features = X_test_raw_thr.shape

        X_train_flat = X_train_raw_thr.reshape(-1, train_features)
        X_val_flat = X_val_raw_thr.reshape(-1, train_features)
        X_test_flat = X_test_raw_thr.reshape(-1, train_features)

        X_train_flat_scaled, X_val_flat_scaled, X_test_flat_scaled, scaler_thr = standardize_data_5min(
            X_train_flat, X_val_flat, X_test_flat
        )

        X_train_thr = X_train_flat_scaled.reshape(train_samples, train_timesteps, train_features)
        X_val_thr = X_val_flat_scaled.reshape(val_samples, val_timesteps, train_features)
        X_test_thr = X_test_flat_scaled.reshape(test_samples, test_timesteps, train_features)

        train_loader_thr = DataLoader(
            TensorDataset(torch.tensor(X_train_thr, dtype=torch.float32), torch.tensor(y_train_thr, dtype=torch.long)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )
        val_loader_thr = DataLoader(
            TensorDataset(torch.tensor(X_val_thr, dtype=torch.float32), torch.tensor(y_val_thr, dtype=torch.long)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )
        test_loader_thr = DataLoader(
            TensorDataset(torch.tensor(X_test_thr, dtype=torch.float32), torch.tensor(y_test_thr, dtype=torch.long)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        )

        for model_name in SWEEP_MODELS:
            print("\n" + "=" * 50)
            print(f"Threshold {threshold} | Model {model_name}")

            model = build_model_by_name(model_name, train_features)
            model.apply(init_weights_safe)

            run_name = f"{model_name}_thr{str(threshold).replace('.', 'p')}"
            lr_value = LR * 0.01 if model_name == 'DLinear' else LR * 0.1
            model_trained, best_val_f1 = safe_train_model_multiclass(
                model,
                run_name,
                train_loader_thr,
                val_loader_thr,
                NUM_CLASSES,
                SWEEP_EPOCHS,
                lr_value,
                patience=SWEEP_PATIENCE,
                use_class_weight=USE_CLASS_WEIGHT_FOR_SWEEP,
            )

            val_logits, val_targets = safe_evaluate_model_multiclass(model_trained, val_loader_thr, NUM_CLASSES)
            test_logits, test_targets = safe_evaluate_model_multiclass(model_trained, test_loader_thr, NUM_CLASSES)

            val_metrics = calculate_multiclass_metrics(val_logits, val_targets, NUM_CLASSES)
            test_metrics = calculate_multiclass_metrics(test_logits, test_targets, NUM_CLASSES)

            sweep_results.append({
                'threshold': threshold,
                'model': model_name,
                'best_val_f1_during_train': float(best_val_f1),
                'val_accuracy': float(val_metrics['accuracy']),
                'val_f1_weighted': float(val_metrics['f1_weighted']),
                'val_f1_macro': float(val_metrics['f1_macro']),
                'test_accuracy': float(test_metrics['accuracy']),
                'test_f1_weighted': float(test_metrics['f1_weighted']),
                'test_f1_macro': float(test_metrics['f1_macro']),
                'test_auc_macro': float(test_metrics['auc_macro']),
                'test_down_f1': float(test_metrics['f1_per_class'][0]),
                'test_neutral_f1': float(test_metrics['f1_per_class'][1]),
                'test_up_f1': float(test_metrics['f1_per_class'][2]),
                'train_neutral_ratio': float(train_stats['neutral_ratio']),
                'val_neutral_ratio': float(val_stats['neutral_ratio']),
                'test_neutral_ratio': float(test_stats['neutral_ratio']),
                'use_class_weight': bool(USE_CLASS_WEIGHT_FOR_SWEEP),
            })

            print(
                f"Result: val_f1_weighted={val_metrics['f1_weighted']:.4f}, "
                f"test_f1_weighted={test_metrics['f1_weighted']:.4f}, "
                f"test_f1_macro={test_metrics['f1_macro']:.4f}"
            )

    print("\n" + "=" * 80)
    print("Threshold sweep completed.")
    print("=" * 80)
else:
    print("No available data. Skip threshold sweep.")



In [ ]:
# ====================== 9. Sweep Summary ======================
print("\n" + "="*80)
print("Step 9: Summarize threshold sweep")
print("="*80)

if 'sweep_results' in locals() and len(sweep_results) > 0:
    sweep_results_df = pd.DataFrame(sweep_results).sort_values(['threshold', 'model'])
    sweep_class_balance_df = pd.DataFrame(sweep_class_balance).sort_values(['threshold', 'split'])

    print("\nModel performance across thresholds:")
    print(sweep_results_df.to_string(index=False))

    print("\nClass balance across thresholds:")
    print(sweep_class_balance_df.to_string(index=False))

    sweep_results_df.to_csv('5min_0405_threshold_sweep_metrics.csv', index=False)
    sweep_class_balance_df.to_csv('5min_0405_threshold_sweep_class_balance.csv', index=False)
    print("\nSaved: 5min_0405_threshold_sweep_metrics.csv")
    print("Saved: 5min_0405_threshold_sweep_class_balance.csv")

    best_per_threshold = (
        sweep_results_df
        .sort_values(['threshold', 'val_f1_weighted'], ascending=[True, False])
        .groupby('threshold', as_index=False)
        .first()
    )

    best_per_threshold['neutral_in_target_train'] = best_per_threshold['train_neutral_ratio'].between(0.4, 0.5)
    best_per_threshold['neutral_in_target_val'] = best_per_threshold['val_neutral_ratio'].between(0.4, 0.5)
    best_per_threshold['neutral_in_target_test'] = best_per_threshold['test_neutral_ratio'].between(0.4, 0.5)

    print("\nBest model per threshold (by val_f1_weighted):")
    print(best_per_threshold.to_string(index=False))

    best_per_threshold.to_csv('5min_0405_threshold_sweep_best_per_threshold.csv', index=False)
    print("Saved: 5min_0405_threshold_sweep_best_per_threshold.csv")

    candidates = best_per_threshold[
        best_per_threshold['neutral_in_target_train'] &
        best_per_threshold['neutral_in_target_val']
    ]

    if len(candidates) > 0:
        recommended = candidates.sort_values('val_f1_weighted', ascending=False).iloc[0]
        print("\nRecommended threshold (meets neutral target 40%-50% on train/val):")
        print(recommended[['threshold', 'model', 'val_f1_weighted', 'test_f1_weighted', 'train_neutral_ratio', 'val_neutral_ratio', 'test_neutral_ratio']])
    else:
        fallback = best_per_threshold.sort_values('val_f1_weighted', ascending=False).iloc[0]
        print("\nNo threshold meets neutral target on both train and val. Best fallback by val_f1_weighted:")
        print(fallback[['threshold', 'model', 'val_f1_weighted', 'test_f1_weighted', 'train_neutral_ratio', 'val_neutral_ratio', 'test_neutral_ratio']])
else:
    print("No sweep results to summarize.")



In [ ]:
# ====================== 10. Sweep Visualization & Export ======================
print("\n" + "="*80)
print("Step 10: Visualize and export threshold-sweep results")
print("="*80)

if 'sweep_results_df' in locals() and len(sweep_results_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    balance_plot_df = sweep_class_balance_df.pivot(index='threshold', columns='split', values='neutral_ratio').sort_index()
    for split in ['train', 'val', 'test']:
        if split in balance_plot_df.columns:
            axes[0].plot(balance_plot_df.index, balance_plot_df[split], marker='o', label=split)
    axes[0].axhline(0.4, color='gray', linestyle='--', linewidth=1)
    axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1)
    axes[0].set_title('Neutral Ratio vs Threshold')
    axes[0].set_xlabel('Threshold')
    axes[0].set_ylabel('Neutral Ratio')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for model_name in sweep_results_df['model'].unique():
        sub = sweep_results_df[sweep_results_df['model'] == model_name].sort_values('threshold')
        axes[1].plot(sub['threshold'], sub['val_f1_weighted'], marker='o', label=model_name)
    axes[1].set_title('Val F1 Weighted vs Threshold')
    axes[1].set_xlabel('Threshold')
    axes[1].set_ylabel('Val F1 Weighted')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    for model_name in sweep_results_df['model'].unique():
        sub = sweep_results_df[sweep_results_df['model'] == model_name].sort_values('threshold')
        axes[2].plot(sub['threshold'], sub['test_f1_macro'], marker='o', label=model_name)
    axes[2].set_title('Test F1 Macro vs Threshold')
    axes[2].set_xlabel('Threshold')
    axes[2].set_ylabel('Test F1 Macro')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('5min_0405_threshold_sweep_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: 5min_0405_threshold_sweep_analysis.png')

    def safe_serialize(obj):
        if isinstance(obj, (np.integer, np.int64, np.int32, np.int16, np.int8)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64, np.float32, np.float16)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.bool_):
            return bool(obj)
        return obj

    payload = {
        'settings': {
            'seq_len': SEQ_LEN,
            'pred_bars': PRED_BARS,
            'sweep_thresholds': SWEEP_THRESHOLDS,
            'sweep_models': SWEEP_MODELS,
            'sweep_epochs': SWEEP_EPOCHS,
            'sweep_patience': SWEEP_PATIENCE,
            'use_class_weight_for_sweep': USE_CLASS_WEIGHT_FOR_SWEEP,
        },
        'results': [{k: safe_serialize(v) for k, v in row.items()} for row in sweep_results],
        'class_balance': [{k: safe_serialize(v) for k, v in row.items()} for row in sweep_class_balance],
    }

    with open('5min_0405_threshold_sweep_results.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print('Saved: 5min_0405_threshold_sweep_results.json')
    print('Done: choose threshold first, then decide whether class weighting is still needed.')
else:
    print('No data to visualize.')

